<a href="https://colab.research.google.com/github/AnastasiaEgorova00/dspracticum2025/blob/main/GPT_libretti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a GPT

This project is a small modification of Andrej Karpathy’s excellent [gpt-dev.ipynb](https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=fjjvMifYZf7x) notebook, which accompanies his video [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY&t=6s) and repo [nanoGPT](https://github.com/karpathy/nanoGPT).


![gpt2](https://www.kapilsharma.dev/assets/gpt2.png)

Image source: https://www.kapilsharma.dev/posts/exploring-gpt2/

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-10-20 13:16:30--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2025-10-20 13:16:30 (102 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



## Before we start building a model

Make sure you name a text file you want to use for training 'input.txt' and upload it to Colab. If the file name is different, modify the cell below.

In [1]:
# read it in to inspect it
with open('sample_data/libretti.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  2386858


In [3]:
# let's look at the first 1000 characters
print(text[:1000])

P R O L O G O
Scena unica
[Tocata]
Ritornello
MUSICA
Dal mio Permesso amato a voi ne vegno,
incliti eroi, sangue gentil di regi,
di cui narra la fama eccelsi pregi,
né giugne al ver perch'è troppo alto il segno.
Io la Musica son, ch'a i dolci accenti
so far tranquillo ogni turbato core,
ed or di nobil ira, ed or d'amore
posso infiammar le più gelate menti.
Io su cetera d'or cantando soglio
mortal orecchio lusingar talora,
e in guisa tal de l'armonia sonora
de le rote del ciel più l'alme invoglio.
Quinci a dirvi d'Orfeo desio mi sprona,
d'Orfeo che trasse al suo cantar le fere,
e servo fe' l'inferno a sue preghiere,
gloria immortal di Pindo e d'Elicona.
Or mentre i canti alterno, or lieti, or mesti,
non si mova augellin fra queste piante,
né s'oda in queste rive onda sonante,
ed ogni auretta in suo camin s'arresti.
Ritornello
A T T O   P R I M O
Scena unica
[Sinfonia]
[Introduzione]
PASTORE (I) In questo lieto e fortunato giorno
ch'ha posto fine a gli amorosi affanni
del nostro semideo,

In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !'(),.0123456789:;?ABCDEFGHILMNOPQRSTUVXYZ[]abcdefghilmnopqrstuvyz~ª«­°º»ÀÈÌÙàèéëìïòùü–‘
91


For now, we won’t perform proper tokenization (it will be your task to train a tokenizer and use it as part of the homework assignment). Instead, we’ll simply tokenize by character.

In [5]:
import tiktoken

# grab some tokenizer
encoding = tiktoken.get_encoding("o200k_base")

In [7]:
textsm = text[:1000]
tokens = encoding.encode(textsm)
print("Encoded tokens:", tokens)

Encoded tokens: [47, 460, 532, 451, 532, 499, 532, 198, 2986, 2995, 168789, 198, 51540, 432, 546, 1592, 49, 278, 2558, 6053, 198, 44, 3042, 29426, 198, 74253, 60428, 20550, 7546, 939, 2754, 261, 30806, 453, 12939, 1750, 412, 103424, 3889, 1111, 3412, 11, 72436, 139565, 1320, 74182, 412, 4091, 30894, 12301, 614, 557, 108110, 55165, 1989, 72, 876, 6248, 412, 10149, 6171, 846, 611, 434, 1245, 78850, 53331, 127601, 25846, 1793, 3055, 1750, 558, 43091, 557, 6619, 1578, 2391, 11, 549, 10443, 575, 10530, 2114, 1259, 14989, 198, 786, 4150, 26426, 16726, 43396, 70346, 2754, 10089, 412, 295, 503, 1320, 297, 10427, 51468, 11, 1648, 503, 272, 30344, 510, 198, 1103, 786, 5603, 72, 27646, 505, 15699, 8063, 379, 12363, 72, 558, 43091, 593, 19505, 2060, 272, 95218, 12538, 1975, 24712, 27142, 198, 76, 48522, 293, 9285, 68386, 305, 1846, 277, 4858, 2505, 412, 68, 306, 1704, 3497, 4858, 334, 305, 6, 2218, 19973, 156811, 198, 613, 505, 84458, 1083, 50853, 15699, 305, 47062, 1047, 1827, 198549, 558, 2231, 

In [8]:
tokens = encoding.encode(text)
print("Encoded tokens:", tokens)

Encoded tokens: [47, 460, 532, 451, 532, 499, 532, 198, 2986, 2995, 168789, 198, 51540, 432, 546, 1592, 49, 278, 2558, 6053, 198, 44, 3042, 29426, 198, 74253, 60428, 20550, 7546, 939, 2754, 261, 30806, 453, 12939, 1750, 412, 103424, 3889, 1111, 3412, 11, 72436, 139565, 1320, 74182, 412, 4091, 30894, 12301, 614, 557, 108110, 55165, 1989, 72, 876, 6248, 412, 10149, 6171, 846, 611, 434, 1245, 78850, 53331, 127601, 25846, 1793, 3055, 1750, 558, 43091, 557, 6619, 1578, 2391, 11, 549, 10443, 575, 10530, 2114, 1259, 14989, 198, 786, 4150, 26426, 16726, 43396, 70346, 2754, 10089, 412, 295, 503, 1320, 297, 10427, 51468, 11, 1648, 503, 272, 30344, 510, 198, 1103, 786, 5603, 72, 27646, 505, 15699, 8063, 379, 12363, 72, 558, 43091, 593, 19505, 2060, 272, 95218, 12538, 1975, 24712, 27142, 198, 76, 48522, 293, 9285, 68386, 305, 1846, 277, 4858, 2505, 412, 68, 306, 1704, 3497, 4858, 334, 305, 6, 2218, 19973, 156811, 198, 613, 505, 84458, 1083, 50853, 15699, 305, 47062, 1047, 1827, 198549, 558, 2231, 

In [10]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[54, 55, 55, 2, 64, 54, 51, 62, 51]
hii there


In [11]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # the first 100 characters of 1000 characters we looked at earier will to the GPT look like this

torch.Size([2386858]) torch.int64
tensor([35,  2, 37,  2, 34,  2, 31,  2, 34,  2, 28,  2, 34,  1, 38, 49, 51, 58,
        47,  2, 65, 58, 55, 49, 47,  1, 45, 39, 59, 49, 47, 64, 47, 46,  1, 37,
        55, 64, 59, 62, 58, 51, 56, 56, 59,  1, 32, 40, 38, 30, 24, 22,  1, 25,
        47, 56,  2, 57, 55, 59,  2, 35, 51, 62, 57, 51, 63, 63, 59,  2, 47, 57,
        47, 64, 59,  2, 47,  2, 66, 59, 55,  2, 58, 51,  2, 66, 51, 53, 58, 59,
         7,  1, 55, 58, 49, 56, 55, 64, 55,  2])


Let us use 90% for training and last 10% for validation.

In [12]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

## Blocks

In [13]:
block_size = 8
train_data[:block_size+1]

tensor([35,  2, 37,  2, 34,  2, 31,  2, 34])

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([35]) the target: 2
when input is tensor([35,  2]) the target: 37
when input is tensor([35,  2, 37]) the target: 2
when input is tensor([35,  2, 37,  2]) the target: 34
when input is tensor([35,  2, 37,  2, 34]) the target: 2
when input is tensor([35,  2, 37,  2, 34,  2]) the target: 31
when input is tensor([35,  2, 37,  2, 34,  2, 31]) the target: 2
when input is tensor([35,  2, 37,  2, 34,  2, 31,  2]) the target: 34


In [15]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 2, 55, 56,  2, 49, 55, 51, 56],
        [51, 49, 59, 58, 50, 47,  1, 26],
        [55, 58,  2, 49, 65, 55,  2, 60],
        [50, 55,  2, 49, 65, 55,  2, 60]])
targets:
torch.Size([4, 8])
tensor([[55, 56,  2, 49, 55, 51, 56,  8],
        [49, 59, 58, 50, 47,  1, 26, 62],
        [58,  2, 49, 65, 55,  2, 60, 65],
        [55,  2, 49, 65, 55,  2, 60, 55]])
----
when input is [2] the target: 55
when input is [2, 55] the target: 56
when input is [2, 55, 56] the target: 2
when input is [2, 55, 56, 2] the target: 49
when input is [2, 55, 56, 2, 49] the target: 55
when input is [2, 55, 56, 2, 49, 55] the target: 51
when input is [2, 55, 56, 2, 49, 55, 51] the target: 56
when input is [2, 55, 56, 2, 49, 55, 51, 56] the target: 8
when input is [51] the target: 49
when input is [51, 49] the target: 59
when input is [51, 49, 59] the target: 58
when input is [51, 49, 59, 58] the target: 50
when input is [51, 49, 59, 58, 50] the target: 47
when input is [51, 49, 

In [16]:
print(xb) # our input to the model

tensor([[ 2, 55, 56,  2, 49, 55, 51, 56],
        [51, 49, 59, 58, 50, 47,  1, 26],
        [55, 58,  2, 49, 65, 55,  2, 60],
        [50, 55,  2, 49, 65, 55,  2, 60]])


## First model

Let’s build the simplest model we can think of.

Some notation:
* **B** — batch size (number of independent sequences processed in parallel)
* **T** — time dimension, number of tokens per sequence
* **C** — number of channels / classes = vocabulary size (the dimensionality of the logits for each token)

In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            # reshaping because of `F.cross_entropy` definition
            logits = logits.view(B*T, C)  # shape (B*T, C)
            targets = targets.view(B*T)   # shape (B*T, )
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# this is before training!
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 91])
tensor(5.1749, grad_fn=<NllLossBackward0>)
r1B'–ºÀGìLOAVìS:vïl,vï–3lBXFU])ªVÀZS8O0lo9l»45T)q‘rUhYTO2EùyqEºQ°Us–ºüpTvùYf
S[9V~oÈZQ2EècïGHÀBª°vvï


In [18]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [19]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


4.954391002655029


In [20]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

?TNÌ»c«º[º75yo.Ùù3ªlO–ÀH;È6!Dfºt3]GLMc~ !dtªCI‘]1DB7à2,ÈèsAagy;T‘:;Xéf–F–àmFëqªCMUOÈZS­ªÈòpé'moïù 3ªìOEìG°BiegÌÙXX5üOhbòm6È‘zB(bO6Ì2«1vIXH8ü?pv0Xv9l[TªLG~55M6OºGVu0ùa–zï‘4È'm­Aè2a7ÙªVC)oè~ªÈUù4Ì:YaVÀhyS–O756'3CDùÌºÈ‘FfDP–zR3àCvùzïB N»s‘yeYYaApFfà
­­utEëü8G°v«o,GÙlgÌ0Cìpë–(gÈZRCé°rtNlssI[
VNºg°E(B3ì[SnYü(òG([TT­2D4Ù]yºMQOBu‘I7PLmQQ;Maé2.v­TCTB4?èCYaUXvy! D?aÙé8O8clÙ6ZébIFMtLÌz
SGéòÈh
i«V~Rueºü‘ceYÌ?Hd0ùzï8ziDév­ga6guSiDPOÌ'mHXe8Q6Bù–ïÀª:3DcH5z~bMYeüDüilÙeTNO8,Gtipcvùa­M(7p8YSqHqì»gàÈüÙEï.–ªH6D:


## Let's improve it a bit

In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 1e-3
max_iters = 1000
eval_interval = 100
eval_iters = 200

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            # reshaping because of `F.cross_entropy` definition
            logits = logits.view(B*T, C)  # shape (B*T, C)
            targets = targets.view(B*T)   # shape (B*T, )
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


0.008281 M parameters
step 0: train loss 5.1237, val loss 5.1035
step 100: train loss 4.9938, val loss 4.9792
step 200: train loss 4.8665, val loss 4.8597
step 300: train loss 4.7400, val loss 4.7323
step 400: train loss 4.6186, val loss 4.6125
step 500: train loss 4.4947, val loss 4.4961
step 600: train loss 4.3853, val loss 4.3842
step 700: train loss 4.2773, val loss 4.2763
step 800: train loss 4.1737, val loss 4.1770
step 900: train loss 4.0741, val loss 4.0787
step 999: train loss 3.9725, val loss 3.9843
3R»2MFÈZIzcEXV.3b)Ly)FCFM)r nBi1avM6c‘5U­4!ù1.TL:AFu0­LDe‘TRM0T]Q1CO~ÈnìàDEoqiVÙcrp.n«­9X1Qv iST4)~°q0ÈDGeIv )Z78»TR«rt6°eo1R«ë'!CDNÀZQantiüü;Àp vP2~Xùdï v[Y0;alRYR8RQàe~22MTP­;ezÌMlg:
8VLQü5ìiPcfAcrºaù2ùvqbVe hYdªd‘fXyÈüà ïì,]z7S(V,­À–ss1GLoò2DfÀM(Q]~qns.;À(ª5ìNì?
é5Z'hYRG[TpX8bichnddqMìie5nL ciegTP,éëé[7ÙXO6IaU0­Y­ªv e'I!;Ov8Müüü–s°'!9)Ù6,CF0È4­Er l;Oì4òàQvefnfloh»Ì–
mg2202H7] qv1Z88fnts.TdDNu!siRZ°ªDG°7–NQBCº3?.NT;B,ECoc Àé~UyU'a,TITXfh»PS!0°8»tÌàR«1EauìªE cies°34~scIl,

## Normalization Layer

In [22]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

## The mathematical trick in self-attention

In [23]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [24]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [25]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [26]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [27]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [28]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [29]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [30]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [31]:
k.var()

tensor(1.0449)

In [32]:
q.var()

tensor(1.0700)

In [33]:
wei.var()

tensor(1.0918)

In [34]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [35]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [36]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(-0.1431), tensor(1.0705))

In [37]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(0.0073), tensor(1.0177))

## Transformer

You may want to refer directly to [the git repo](https://github.com/karpathy/ng-video-lecture) instead though.

In [38]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

# for saving the model
import os, json
SAVE_DIR = "./libretti_final_model"
MODEL_PATH = os.path.join(SAVE_DIR, "model_final.pt")
META_PATH = os.path.join(SAVE_DIR, "meta.json")
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(1337)



# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.213083 M parameters
step 0: train loss 4.6706, val loss 4.6693
step 100: train loss 2.6495, val loss 2.7219
step 200: train loss 2.4863, val loss 2.5347
step 300: train loss 2.4223, val loss 2.4639
step 400: train loss 2.3841, val loss 2.4266
step 500: train loss 2.3162, val loss 2.3619
step 600: train loss 2.2625, val loss 2.3172
step 700: train loss 2.2182, val loss 2.2814
step 800: train loss 2.1814, val loss 2.2468
step 900: train loss 2.1538, val loss 2.2195
step 1000: train loss 2.1285, val loss 2.1976
step 1100: train loss 2.1160, val loss 2.1792
step 1200: train loss 2.0979, val loss 2.1510
step 1300: train loss 2.0785, val loss 2.1486
step 1400: train loss 2.0658, val loss 2.1280
step 1500: train loss 2.0558, val loss 2.1287
step 1600: train loss 2.0356, val loss 2.1098
step 1700: train loss 2.0181, val loss 2.0909
step 1800: train loss 2.0153, val loss 2.0929
step 1900: train loss 2.0006, val loss 2.0697
step 2000: train loss 1.9859, val loss 2.0677
step 2100: train loss 1.

In [39]:
def save_meta():
    meta = {
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
        "chars": chars,             # to rebuild vocab exactly
    }
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

save_meta()
torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved final model to {MODEL_PATH} and meta to {META_PATH}")

Saved final model to ./libretti_final_model/model_final.pt and meta to ./libretti_final_model/meta.json


## Using the saved model

The following code recreates the model and reads saved weights.

In [41]:
# - Assumes you trained & saved with:
#   - MODEL:  ./libretti_final_model/model_final.pt
#   - META:   ./libretti_final_model/meta.json

import torch
import torch.nn as nn
from torch.nn import functional as F
import os, json

SAVE_DIR = "./libretti_final_model"
MODEL_PATH = os.path.join(SAVE_DIR, "model_final.pt")
META_PATH = os.path.join(SAVE_DIR, "meta.json")

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (os.path.exists(MODEL_PATH) and os.path.exists(META_PATH)):
    raise FileNotFoundError(f"Missing model or meta file in {SAVE_DIR}")

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

block_size = meta["block_size"]
n_embd     = meta["n_embd"]
n_head     = meta["n_head"]
n_layer    = meta["n_layer"]
dropout    = meta["dropout"]
chars      = meta["chars"]

vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

# ---------- model definition ----------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedFoward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------- load weights ----------
model = BigramLanguageModel().to(device)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval()

# ---------- generate ----------
seed_text = ""  # you can add a prompt here if you want
if seed_text:
    for c in seed_text:
        if c not in stoi:
            raise ValueError(f"Out-of-vocab char in seed: {repr(c)}")
    start = torch.tensor([encode(seed_text)], dtype=torch.long, device=device)
else:
    start = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(start, max_new_tokens=2000)[0].tolist()
print(decode(generated))

esto in dal queuRe non fuge sombra.
VASTOPE Ah, luce ad acità!
O riva dal mopo io fere?
Geniatino,
che sei d'infessiglio,
riballo. Come
figlo io con sava fecio,
m'affiosa il puer setta
voltate il pur ciel
fior condegno arregiato.
MERICLLO E la l'occun mene
mi portuno, e sche;
mentoscan onoscendola.
TITONFIA E che di costai scassidia
ha tua pea?
E par ch'io chi spugge
io pur restenze quanto che dinguira,
dolentenza setelle.
IGICA Ohimè, piùché sui chiote,
sompri, e le riccigne?
E radite perdone
non io s'uspete;
dall'ornon queste premente voi?
Riverero, è vostra.
Core
scupuri iomai rappo il monto,
sche sua costente, i fanne
non quest'indoli, e non scrrai?
Che della stua beltue songue
d'el lumeta volente me.
Per che lo chiene
spettri o pa',
pur dispivosolo
l'infesse e tramese ol coro punua,
deh vincome, o moriri
daperi ancime dabbiaci;
e ben dentra d'irmane
che questi il tuo vallo eternoso.
Scena fesorra, e 'l coste pene.
LITORCA Mebrà, ch'è pugnito
in regina amicito?
ALINDO Poiane,
che 